[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/11_sliding_window.ipynb)

# 🔴 Hard: Sliding Window Attention

Implement **Sliding Window Attention** — used in Longformer, Mistral, etc. for efficient long-context processing.

Each position $i$ can only attend to positions $j$ where $|i - j| \le w$ (the window size).

### Signature
```python
def sliding_window_attention(Q, K, V, window_size):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
    # window_size: int — position i attends to [i-w, i+w]
```

### Rules
- Do **NOT** use sparse attention libraries
- Mask positions outside the window with `-inf`
- `window_size=0`: only self — output should equal V
- `window_size >= seq_len`: equivalent to full attention

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch
import math

/usr/local/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [5]:
# ✏️ YOUR IMPLEMENTATION HERE
def sliding_window_attention(Q, K, V, window_size):
    batch_size, seq_len, d_q = Q.shape
    _, _, d_v = V.shape
    atten_score = Q @ K.transpose(1,2)
    mask = torch.tril(torch.ones(seq_len, seq_len), window_size)
    mask = torch.triu(mask, -window_size)
    atten_score = atten_score.masked_fill(mask == 0, float('-inf'))
    atten_score = torch.softmax(atten_score / math.sqrt(d_q), dim = -1)
    return atten_score @ V

In [6]:
# 🧪 Debug
Q = torch.randn(1, 6, 8)
K = torch.randn(1, 6, 8)
V = torch.randn(1, 6, 8)

out = sliding_window_attention(Q, K, V, window_size=1)
print("Output shape:", out.shape)  # (1, 6, 8)

# window=0 should return V
out0 = sliding_window_attention(Q, K, V, window_size=0)
print("window=0 == V?", torch.allclose(out0, V, atol=1e-5))

Output shape: torch.Size([1, 6, 8])
window=0 == V? True


In [7]:
from torch_judge import check
check('sliding_window')


🧪 Testing: Sliding Window Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (1.4ms)
  ✅ [2/5] window_size=0 — only sees itself (0.5ms)
  ✅ [3/5] Large window equals full attention (1.1ms)
  ✅ [4/5] Distant tokens don't affect output (1.1ms)
  ✅ [5/5] Gradient flow (0.6ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (4.6ms total)
  Progress saved. Run status() to see your dashboard.

